## 1 — Imports & Setup

In [1]:
import os, json, random, numpy as np, pandas as pd, torch, torch.nn as nn, cv2
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms
import subprocess, sys

try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'networkx'])
    import networkx as nx

DATA_PATH   = '/kaggle/input/competitions/ml-ware-26-sherlock-files/Evidence_Motion_Records_MLWare_26 (2)/Evidence_Motion_Records_MLWare_26'
TRAIN_PATH  = f'{DATA_PATH}/train'
TEST_PATH   = f'{DATA_PATH}/test'
LABEL_PATH  = f'{DATA_PATH}/train_labels.json'
FEATURE_DIR = '/kaggle/working/features'   # writable path
MOTION_DIR  = '/kaggle/working/motion'     # cache motion features
SUB_PATH    = '/kaggle/working/submission.csv'
WEIGHTS_PATH= '/kaggle/working/model_weights.pth'

os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(MOTION_DIR,  exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Train  : {len(os.listdir(TRAIN_PATH))} videos')
print(f'Test   : {len(os.listdir(TEST_PATH))} videos')


Device : cuda
Train  : 5611 videos
Test   : 296 videos


## 2 — Load DINOv2

In [2]:
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dino = dino.to(DEVICE).eval()
print('DINOv2 loaded')


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 288MB/s]


DINOv2 loaded


## 3 — Frame & Feature Utilities

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def count_frames(video_path):
    """True frame count without full decode. Falls back to counting reads."""
    cap = cv2.VideoCapture(video_path)
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if n > 0:
        return n
    cap = cv2.VideoCapture(video_path)
    c = 0
    while True:
        ok, _ = cap.read()
        if not ok: break
        c += 1
    cap.release()
    return c

def extract_frames(video_path):
    cap, frames = cv2.VideoCapture(video_path), []
    while True:
        ok, frame = cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def get_features(frames, batch_size=32):
    feats = []
    for i in range(0, len(frames), batch_size):
        imgs = [Image.fromarray(f) for f in frames[i:i+batch_size]]
        x = torch.stack([transform(img) for img in imgs]).to(DEVICE)
        with torch.no_grad():
            feats.append(dino(x).cpu().numpy())
    return np.concatenate(feats)   # (N, 384)

print('Utilities ready')


Utilities ready


## 4 — Physics-Aware Motion Feature Extraction (NEW)

In [4]:
# ── Motion feature constants ───────────────────────────────────────────────
FLOW_RESIZE  = (64, 64)   # small size — fast Farneback, low memory
FLOW_GAPS    = [1, 3, 5]  # multi-scale temporal gaps
MOTION_DIM   = 6          # per-gap: fx_mean, fy_mean, mag_mean, mag_std, diff_mean, diff_std
MOTION_TOTAL = len(FLOW_GAPS) * MOTION_DIM   # 18-dim total


def compute_optical_flow_features(frame_a, frame_b):
    """
    Compute Farneback optical flow between two RGB frames.
    Returns 6-dim motion vector:
        [flow_x_mean, flow_y_mean, mag_mean, mag_std, diff_mean, diff_std]
    Memory safe: works on small resized grays.
    """
    # Resize to small size for speed & memory safety
    a = cv2.resize(frame_a, FLOW_RESIZE)
    b = cv2.resize(frame_b, FLOW_RESIZE)
    ga = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gb = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # Farneback optical flow — lightweight, no GPU needed
    flow = cv2.calcOpticalFlowFarneback(
        ga, gb, None,
        pyr_scale=0.5, levels=2, winsize=8,
        iterations=2, poly_n=5, poly_sigma=1.1,
        flags=0
    )   # (H, W, 2)

    fx, fy = flow[..., 0], flow[..., 1]
    mag    = np.sqrt(fx**2 + fy**2)

    # Frame pixel difference
    diff = np.abs(ga - gb)

    return np.array([
        fx.mean(), fy.mean(),
        mag.mean(), mag.std(),
        diff.mean(), diff.std()
    ], dtype=np.float32)


def compute_multiscale_motion(frames, idx, gaps=FLOW_GAPS):
    """
    Compute motion features for frame at idx using multiple temporal gaps.
    For gap g: compare frame[idx] with frame[idx+g] (clipped to valid range).
    Returns (18,) float32 array.
    """
    n = len(frames)
    parts = []
    for g in gaps:
        j = min(idx + g, n - 1)   # clip to last valid frame
        parts.append(compute_optical_flow_features(frames[idx], frames[j]))
    return np.concatenate(parts)   # (18,)


def build_motion_cache(frames):
    """
    Pre-compute multiscale motion for every frame in a video.
    Returns (N, 18) numpy array.
    Cached per video to avoid recomputing during pair sampling.
    """
    N = len(frames)
    cache = np.zeros((N, MOTION_TOTAL), dtype=np.float32)
    for i in range(N):
        cache[i] = compute_multiscale_motion(frames, i)
    return cache


def get_pairwise_motion(motion_cache, i, j):
    """
    Pairwise motion feature for pair (i, j).
    Use frame i's own multi-scale motion as the motion context for the pair.
    Returns (18,) float32 — represents velocity leaving frame i.
    """
    return motion_cache[i]   # (18,)


print(f'Motion utilities ready  |  motion_dim={MOTION_TOTAL}')


Motion utilities ready  |  motion_dim=18


## 5 — Extract & Cache DINOv2 Features

In [5]:
with open(LABEL_PATH) as f:
    labels = json.load(f)   # keys: 'video_X' (no .mp4), values: 1-indexed order list
print(f'Labels: {len(labels)}')

train_videos = os.listdir(TRAIN_PATH)

for vid in tqdm(train_videos, desc='Caching DINOv2 features'):
    save_path = os.path.join(FEATURE_DIR, f'{vid}.npy')
    if os.path.exists(save_path): continue
    frames = extract_frames(os.path.join(TRAIN_PATH, vid))
    if not frames: continue
    feats = get_features(frames)
    np.save(save_path, feats)

print(f'Features cached: {len(os.listdir(FEATURE_DIR))} files')


Labels: 5611


Caching DINOv2 features:  71%|███████   | 3981/5611 [31:52<13:02,  2.08it/s][mpeg4 @ 0x16965380] ac-tex damaged at 19 6
[mpeg4 @ 0x16965380] Error at MB: 145
[mpeg4 @ 0x158f8980] ac-tex damaged at 4 6
[mpeg4 @ 0x158f8980] Error at MB: 130
[mpeg4 @ 0x209b5780] ac-tex damaged at 11 7
[mpeg4 @ 0x209b5780] Error at MB: 158
[mpeg4 @ 0x17fcf840] ac-tex damaged at 10 6
[mpeg4 @ 0x17fcf840] Error at MB: 136
[mpeg4 @ 0x16965380] header damaged
[mpeg4 @ 0x158f8980] ac-tex damaged at 5 7
[mpeg4 @ 0x158f8980] Error at MB: 152
[mpeg4 @ 0x17fcf840] ac-tex damaged at 7 7
[mpeg4 @ 0x17fcf840] Error at MB: 154
[mpeg4 @ 0x158f8980] ac-tex damaged at 14 7
[mpeg4 @ 0x158f8980] Error at MB: 161
Caching DINOv2 features: 100%|██████████| 5611/5611 [45:02<00:00,  2.08it/s]

Features cached: 5611 files


## 6 — Extract & Cache Motion Features (NEW)

In [6]:
for vid in tqdm(train_videos, desc='Caching motion features'):
    save_path = os.path.join(MOTION_DIR, f'{vid}.npy')
    if os.path.exists(save_path): continue
    frames = extract_frames(os.path.join(TRAIN_PATH, vid))
    if not frames: continue
    motion = build_motion_cache(frames)   # (N, 18)
    np.save(save_path, motion)

print(f'Motion cached: {len(os.listdir(MOTION_DIR))} files')


Caching motion features:  71%|███████   | 3981/5611 [28:39<11:38,  2.33it/s][mpeg4 @ 0x16965380] ac-tex damaged at 19 6
[mpeg4 @ 0x16965380] Error at MB: 145
[mpeg4 @ 0x209b5780] ac-tex damaged at 4 6
[mpeg4 @ 0x209b5780] Error at MB: 130
[mpeg4 @ 0x17e30540] ac-tex damaged at 11 7
[mpeg4 @ 0x17e30540] Error at MB: 158
[mpeg4 @ 0x156c6b00] ac-tex damaged at 10 6
[mpeg4 @ 0x156c6b00] Error at MB: 136
[mpeg4 @ 0x16965380] header damaged
[mpeg4 @ 0x209b5780] ac-tex damaged at 5 7
[mpeg4 @ 0x209b5780] Error at MB: 152
[mpeg4 @ 0x156c6b00] ac-tex damaged at 7 7
[mpeg4 @ 0x156c6b00] Error at MB: 154
[mpeg4 @ 0x209b5780] ac-tex damaged at 14 7
[mpeg4 @ 0x209b5780] Error at MB: 161
Caching motion features: 100%|██████████| 5611/5611 [40:27<00:00,  2.31it/s]

Motion cached: 5611 files


## 7 — Physics-Aware Pairwise Model (UPDATED)

In [7]:
# Input dim: feat_i(384) + feat_j(384) + diff(384) + motion_ij(18) = 1170
FEAT_DIM   = 384
INPUT_DIM  = FEAT_DIM * 3 + MOTION_TOTAL   # 1170

class TemporalModel(nn.Module):
    """
    Physics-aware pairwise temporal classifier.
    Input : [feat_i | feat_j | feat_j-feat_i | motion_ij]  (1170-dim)
    Output: P(frame i comes before frame j)

    No positional encoding — frames are shuffled, position is meaningless.
    """
    def __init__(self, in_dim=INPUT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512,    256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256,     64), nn.ReLU(),
            nn.Linear(64,       1), nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x)

model     = TemporalModel().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)
bce_loss  = nn.BCELoss()
print(f'Params    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Input dim : {INPUT_DIM}')


Params    : 747,393
Input dim : 1170


## 8 — Physics Consistency Loss & Motion-Aware Sampler (NEW)

In [8]:
LAMBDA_PHYSICS = 0.1   # weight for physics consistency loss
MAX_PAIRS      = 300   # pairs per video (memory safe)


def physics_consistency_loss(motion_cache, triplets):
    """
    For triplets (A, B, C): flow(A,B) + flow(B,C) ≈ flow(A,C)
    L_physics = mean ||flow_AB + flow_BC - flow_AC||^2

    Uses only the first 6 dims (gap-1 motion) for the consistency check.
    triplets: list of (a, b, c) index tuples
    """
    if len(triplets) == 0:
        return torch.tensor(0.0, device=DEVICE)

    losses = []
    for a, b, c in triplets:
        # Use gap-1 motion (first 6 dims)
        flow_ab = motion_cache[a, :6]
        flow_bc = motion_cache[b, :6]
        flow_ac = motion_cache[a, :6]   # approx: reuse frame-a motion
        residual = flow_ab + flow_bc - flow_ac
        losses.append((residual ** 2).mean())

    return torch.tensor(np.mean(losses), dtype=torch.float32, device=DEVICE)


def motion_aware_pair_sampler(n, motion_cache, max_pairs=MAX_PAIRS):
    """
    Sample frame pairs with probability proportional to motion magnitude.
    Pairs with larger motion give stronger temporal cues.
    Falls back to uniform sampling if motion is all zero.
    """
    all_pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
    if len(all_pairs) <= max_pairs:
        return all_pairs

    # Weight by mean motion magnitude of both frames in the pair
    weights = np.array([
        motion_cache[i, 2] + motion_cache[j, 2]   # mag_mean for gap-1
        for i, j in all_pairs
    ], dtype=np.float32)

    total = weights.sum()
    if total < 1e-8:
        # All-zero motion — fall back to uniform
        return random.sample(all_pairs, max_pairs)

    probs = weights / total
    idxs  = np.random.choice(len(all_pairs), size=max_pairs, replace=False, p=probs)
    return [all_pairs[k] for k in idxs]


def build_pairwise_features(features, motion_cache, pairs):
    """
    Build physics-aware pairwise feature matrix.
    Each row: [feat_i | feat_j | feat_j-feat_i | motion_ij]  (1170-dim)
    """
    X, y_dummy = [], []
    for i, j in pairs:
        fi   = features[i]
        fj   = features[j]
        diff = fj - fi
        mot  = get_pairwise_motion(motion_cache, i, j)
        X.append(np.concatenate([fi, fj, diff, mot]))
    return np.array(X, dtype=np.float32)


print('Physics loss & motion sampler ready')
print(f'lambda_physics={LAMBDA_PHYSICS}  max_pairs={MAX_PAIRS}')


Physics loss & motion sampler ready
lambda_physics=0.1  max_pairs=300


## 9 — Training with Physics Consistency Loss (UPDATED)

In [9]:
EPOCHS     = 8
feat_files = [f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy')]
print(f'Training on {len(feat_files)} videos  x  {EPOCHS} epochs')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_phys, count = 0.0, 0.0, 0
    random.shuffle(feat_files)

    for vid_file in tqdm(feat_files, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        vid_key = vid_file.replace('.mp4.npy', '')
        if vid_key not in labels: continue

        # Load DINOv2 features
        features = np.load(os.path.join(FEATURE_DIR, vid_file))   # (N, 384)

        # Load motion features (skip video if motion not cached)
        motion_path = os.path.join(MOTION_DIR, vid_file)
        if not os.path.exists(motion_path): continue
        motion_cache = np.load(motion_path)                        # (N, 18)

        # Labels: 1-indexed in JSON -> 0-indexed
        order = [x - 1 for x in labels[vid_key]]

        # Reconcile length mismatch
        N, L = len(features), len(order)
        if N != L:
            if abs(N - L) <= 3:
                features     = features[:L]
                motion_cache = motion_cache[:L]
                order        = order[:len(features)]
            else:
                continue

        n = len(features)
        pos = {frame_idx: rank for rank, frame_idx in enumerate(order)}

        # Motion-aware pair sampling
        pairs = motion_aware_pair_sampler(n, motion_cache, MAX_PAIRS)
        if len(pairs) == 0: continue

        # Build feature matrix and labels
        X = build_pairwise_features(features, motion_cache, pairs)
        y = np.array([
            1.0 if pos.get(i, 0) < pos.get(j, 0) else 0.0
            for i, j in pairs
        ], dtype=np.float32)

        X_t = torch.tensor(X).to(DEVICE)
        y_t = torch.tensor(y).unsqueeze(1).to(DEVICE)

        # Physics consistency loss on random triplets (cheap: max 20 triplets)
        idxs    = list(range(n))
        triplets = []
        if n >= 3:
            for _ in range(min(20, n)):
                a, b, c = random.sample(idxs, 3)
                triplets.append((a, b, c))

        optimizer.zero_grad()
        pred      = model(X_t)
        loss_bce  = bce_loss(pred, y_t)
        loss_phys = physics_consistency_loss(motion_cache, triplets)
        loss      = loss_bce + LAMBDA_PHYSICS * loss_phys
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Memory cleanup every 50 videos
        if count % 50 == 0:
            del X_t, y_t
            torch.cuda.empty_cache()

        total_loss += loss_bce.item()
        total_phys += loss_phys.item()
        count      += 1

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{EPOCHS}  bce={total_loss/max(count,1):.4f}  '
          f'phys={total_phys/max(count,1):.4f}  lr={lr:.2e}')

model.eval()
torch.save(model.state_dict(), WEIGHTS_PATH)
print(f'Training done. Weights saved -> {WEIGHTS_PATH}')


Training on 5611 videos  x  8 epochs


Epoch 01/8  bce=0.6942  phys=81.1325  lr=2.89e-04


Epoch 02/8  bce=0.6916  phys=82.8476  lr=2.56e-04


Epoch 03/8  bce=0.6874  phys=81.8729  lr=2.07e-04


Epoch 04/8  bce=0.6805  phys=81.9881  lr=1.50e-04


Epoch 05/8  bce=0.6698  phys=80.9061  lr=9.26e-05


Epoch 06/8  bce=0.6563  phys=81.4057  lr=4.39e-05


Epoch 07/8  bce=0.6434  phys=81.8638  lr=1.14e-05


Epoch 08/8  bce=0.6344  phys=81.1186  lr=0.00e+00
Training done. Weights saved -> /kaggle/working/model_weights.pth


## 10 — Inference: Graph Ordering + Beam Search Refinement (UPDATED)

In [10]:
BEAM_SIZE = 5   # beam search width for temporal smoothness refinement


def predict_order_graph(features, motion_cache):
    """
    Build pairwise graph using physics-aware features.
    Returns 0-indexed order, length always == len(features).
    """
    n = len(features)
    if n == 1:
        return [0]

    pairs = [(i, j) for i in range(n) for j in range(i+1, n)]

    # Build feature matrix in batches to stay memory safe (batch <= 16)
    INFER_BATCH = 2048   # pairs per forward pass
    probs_all   = []

    for start in range(0, len(pairs), INFER_BATCH):
        batch_pairs = pairs[start:start+INFER_BATCH]
        rows = []
        for i, j in batch_pairs:
            fi   = features[i]; fj = features[j]
            mot  = get_pairwise_motion(motion_cache, i, j)
            rows.append(np.concatenate([fi, fj, fj-fi, mot]))
        X = torch.tensor(np.array(rows, dtype=np.float32)).to(DEVICE)
        with torch.no_grad():
            probs_all.append(model(X).squeeze(1).cpu().numpy())
        del X
        torch.cuda.empty_cache()

    probs = np.concatenate(probs_all)

    # Build directed graph
    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    for (i, j), p in zip(pairs, probs):
        if p > 0.5: G.add_edge(i, j, weight=float(p))
        else:       G.add_edge(j, i, weight=float(1-p))

    try:
        order = list(nx.topological_sort(G))
    except Exception:
        order = sorted(range(n), key=lambda x: -G.out_degree(x))

    # Guarantee completeness
    seen    = set(order)
    missing = [i for i in range(n) if i not in seen]
    return order + missing


def motion_penalty(seq, motion_cache):
    """
    Temporal smoothness penalty for a candidate sequence.
    penalty = sum ||motion(i,i+1) - motion(i+1,i+2)||  for consecutive triples.
    Lower is smoother / more physically consistent.
    Uses gap-1 motion (first 6 dims) only.
    """
    penalty = 0.0
    for k in range(len(seq) - 2):
        a, b, c = seq[k], seq[k+1], seq[k+2]
        m_ab = motion_cache[a, :6]
        m_bc = motion_cache[b, :6]
        penalty += np.linalg.norm(m_ab - m_bc)
    return penalty


def beam_search_refinement(base_order, motion_cache, beam_size=BEAM_SIZE):
    """
    Refine base_order using beam search over local swaps.
    Considers swapping adjacent pairs to reduce motion_penalty.
    Beam size limits candidates at each step — stays lightweight.

    Returns refined order (0-indexed).
    """
    n = len(base_order)
    if n <= 2:
        return base_order

    # Start beam with the base order
    beam = [list(base_order)]

    for step in range(min(n - 1, 20)):   # max 20 refinement steps
        candidates = []
        for seq in beam:
            candidates.append(seq)   # keep current
            # Try all adjacent swaps
            for k in range(len(seq) - 1):
                swapped = seq[:]
                swapped[k], swapped[k+1] = swapped[k+1], swapped[k]
                candidates.append(swapped)

        # Score all candidates, keep best beam_size
        scored = sorted(candidates, key=lambda s: motion_penalty(s, motion_cache))
        # Deduplicate
        seen_keys = set()
        beam = []
        for s in scored:
            key = tuple(s)
            if key not in seen_keys:
                seen_keys.add(key)
                beam.append(s)
            if len(beam) >= beam_size:
                break

    return beam[0]   # best candidate


print(f'Inference utilities ready  |  beam_size={BEAM_SIZE}')


Inference utilities ready  |  beam_size=5


## 11 — Inference on Test Set

In [11]:
test_videos = sorted(f for f in os.listdir(TEST_PATH) if f.endswith('.mp4'))
print(f'Running inference on {len(test_videos)} test videos...')

results = []

for vid in tqdm(test_videos, desc='Inference'):
    vid_path = os.path.join(TEST_PATH, vid)
    vid_id   = vid.replace('.mp4', '')

    # True frame count — what portal checks against
    true_len = count_frames(vid_path)
    frames   = extract_frames(vid_path)
    actual_len = len(frames)

    if actual_len == 0:
        order_str = '[' + ','.join(str(i+1) for i in range(max(true_len, 1))) + ']'
        results.append([vid_id, order_str])
        continue

    # DINOv2 features
    feats = get_features(frames)                     # (actual_len, 384)

    # Motion features (computed on-the-fly for test)
    motion_cache = build_motion_cache(frames)        # (actual_len, 18)

    # Graph-based ordering
    order = predict_order_graph(feats, motion_cache) # 0-indexed, len=actual_len

    # Beam search temporal refinement
    order = beam_search_refinement(order, motion_cache)

    # Align to true_len — prevents portal frame count mismatch error
    if true_len > actual_len:
        seen    = set(order)
        missing = [i for i in range(true_len) if i not in seen]
        order   = order + missing
    order = order[:true_len]

    # Safety net
    if len(order) != true_len or len(set(order)) != true_len:
        order = list(range(true_len))

    # 0-indexed -> 1-indexed for submission
    order_str = '[' + ','.join(str(x+1) for x in order) + ']'
    results.append([vid_id, order_str])

    # Periodic memory cleanup
    del feats, motion_cache
    torch.cuda.empty_cache()

print(f'Done: {len(results)} videos')


Running inference on 296 test videos...


Inference: 100%|██████████| 296/296 [25:13<00:00,  5.11s/it]

Done: 296 videos


## 12 — Save Submission CSV

In [12]:
df = pd.DataFrame(results, columns=['ID', 'order'])
df['_n'] = df['ID'].str.extract(r'(\d+)$').astype(int)
df = df.sort_values('_n').drop(columns='_n').reset_index(drop=True)
df.to_csv(SUB_PATH, index=False)

print(f'Saved : {SUB_PATH}')
print(f'Rows  : {len(df)}')
print(df.head(3).to_string())
print()

# Hard format assertions — crash loudly before a bad submission
for _, row in df.iterrows():
    o    = row['order']
    assert o.startswith('[') and o.endswith(']'), f"Bad format: {row['ID']}"
    vals = list(map(int, o[1:-1].split(',')))
    assert min(vals) >= 1,              f"0-indexed value in {row['ID']}: min={min(vals)}"
    assert len(vals)==len(set(vals)),   f"Duplicate indices in {row['ID']}"

print('All format assertions passed. Safe to submit.')


Saved : /kaggle/working/submission.csv
Rows  : 296
           ID                                                                                                                                                                                                                                                                                                                                                                                          order
0  video_5611                                                                                          [60,57,59,58,62,54,53,63,95,55,2,11,26,52,65,25,66,61,27,43,10,51,50,9,28,49,14,24,42,47,94,41,44,92,93,96,36,16,91,46,56,15,97,87,37,98,100,17,35,22,88,99,19,3,5,86,89,18,90,20,13,33,6,45,48,68,80,21,85,79,81,82,30,32,75,67,70,40,69,74,31,23,38,78,83,84,29,73,76,77,34,39,71,4,8,12,64,1,7,72]
1  video_5612  [13,15,16,67,17,51,52,14,43,53,12,61,39,33,50,56,48,54,55,46,44,30,47,45,11,58,5,57,59,32,60,104,49,106,108,107,10,62,68,95,109,93,63,35,92,